# Mean Shift Ablation Study for State Model


## Objective

This notebook tests whether a simple **mean shift baseline** can match the performance of the full State transformer model. The goal is to understand:

1. **How much predictive power comes from learning the "average shift" per perturbation?**
2. **At what training epoch does the transformer surpass the mean shift baseline?**
3. **Can we initialize training with mean shifts to speed up convergence?**

## Hypothesis

The perturbation effect can be approximated as a **consistent shift in embedding space** for each (cell_type, perturbation) pair:

```
perturbed_embedding ≈ control_embedding + mean_shift
```

If this approximation is good, the mean shift baseline should achieve similar performance to the State transformer.

## Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import pickle
from pathlib import Path
from tqdm import tqdm

# Import our mean shift implementation
from mean_shift_ablation import MeanShiftTable, evaluate_mean_shift_baseline

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("Imports successful!")

## Data Configuration

Specify the paths to your training and test data, and the column names in your AnnData objects.

In [ ]:
# ============================================================================
# CONFIGURATION - MODIFY THESE PATHS FOR YOUR DATA
# ============================================================================

# Data paths
TRAIN_DATA_PATH = "competition_support_set/train_data.h5ad"  # TODO: Update this path
TEST_DATA_PATH = "competition_support_set/test_data.h5ad"    # TODO: Update this path

# Column names in adata.obs
CELL_TYPE_COL = "cell_type"        # Column containing cell type labels
PERT_COL = "target_gene"           # Column containing perturbation names
CONTROL_PERT = "non-targeting"     # Name of control/untreated perturbation

# Embedding location
EMBED_KEY = "X_hvg"                # Key in adata.obsm for embeddings, or None for adata.X

# Output paths
OUTPUT_DIR = Path("mean_shift_results")
OUTPUT_DIR.mkdir(exist_ok=True)

print(f"Configuration:")
print(f"  Train data: {TRAIN_DATA_PATH}")
print(f"  Test data: {TEST_DATA_PATH}")
print(f"  Cell type column: {CELL_TYPE_COL}")
print(f"  Perturbation column: {PERT_COL}")
print(f"  Control perturbation: {CONTROL_PERT}")
print(f"  Embedding key: {EMBED_KEY}")
print(f"  Output directory: {OUTPUT_DIR}")

## Step 1: Load and Inspect Training Data

First, let's load the training data and verify its structure.

In [ ]:
print("Loading training data...")
adata_train = sc.read_h5ad(TRAIN_DATA_PATH)

print(f"\nTraining data shape: {adata_train.shape}")
print(f"  - Cells: {adata_train.n_obs:,}")
print(f"  - Features: {adata_train.n_vars:,}")

print(f"\nColumns in adata.obs: {list(adata_train.obs.columns)}")
print(f"Keys in adata.obsm: {list(adata_train.obsm.keys())}")

# Get embeddings
if EMBED_KEY and EMBED_KEY in adata_train.obsm:
    train_embeddings = adata_train.obsm[EMBED_KEY]
    print(f"\nUsing embeddings from adata.obsm['{EMBED_KEY}']")
else:
    train_embeddings = adata_train.X.toarray() if hasattr(adata_train.X, 'toarray') else adata_train.X
    print(f"\nUsing embeddings from adata.X")

print(f"Embedding shape: {train_embeddings.shape}")
embedding_dim = train_embeddings.shape[1]

# Verify required columns exist
assert CELL_TYPE_COL in adata_train.obs.columns, f"Column '{CELL_TYPE_COL}' not found in adata.obs"
assert PERT_COL in adata_train.obs.columns, f"Column '{PERT_COL}' not found in adata.obs"

# Get unique values
unique_cell_types = adata_train.obs[CELL_TYPE_COL].unique()
unique_perts = adata_train.obs[PERT_COL].unique()

print(f"\nUnique cell types: {len(unique_cell_types)}")
print(f"Cell types: {list(unique_cell_types)}")

print(f"\nUnique perturbations: {len(unique_perts)}")
print(f"First 10 perturbations: {list(unique_perts[:10])}")

# Check for control perturbation
assert CONTROL_PERT in unique_perts, f"Control perturbation '{CONTROL_PERT}' not found in data"
n_control = (adata_train.obs[PERT_COL] == CONTROL_PERT).sum()
print(f"\nControl cells ('{CONTROL_PERT}'): {n_control:,} ({n_control/adata_train.n_obs*100:.1f}%)")

## Step 2: Compute Mean Shift Table

For each (cell_type, perturbation) pair, we compute:

```python
control_mean = mean(control_embeddings for this cell_type)
perturbed_mean = mean(perturbed_embeddings for this cell_type + perturbation)
shift = perturbed_mean - control_mean
```

This shift vector captures the "average effect" of applying that perturbation to that cell type.

In [ ]:
print("Computing mean shift table...")

shift_table = MeanShiftTable()
shift_table.compute_from_anndata(
    adata_path=TRAIN_DATA_PATH,
    control_pert=CONTROL_PERT,
    cell_type_col=CELL_TYPE_COL,
    pert_col=PERT_COL,
    embed_key=EMBED_KEY
)

# Save the shift table
shift_table_path = OUTPUT_DIR / "mean_shift_table.pkl"
shift_table.save(str(shift_table_path))

print(f"\nMean shift table saved to: {shift_table_path}")

## Step 3: Verify Shift Table Structure

Let's verify that the computed shifts have the correct dimensions and structure.

In [ ]:
print("="*60)
print("SHIFT TABLE VERIFICATION")
print("="*60)

# Check number of shifts computed
n_shifts = len(shift_table.shifts)
n_control_means = len(shift_table.control_means)

print(f"\nNumber of shifts computed: {n_shifts}")
print(f"Number of cell types with control means: {n_control_means}")

# Verify all shifts have correct dimensions
print(f"\nVerifying shift dimensions...")
for key, shift in shift_table.shifts.items():
    assert shift.shape == (embedding_dim,), f"Shift {key} has wrong shape: {shift.shape}"
print(f"✓ All {n_shifts} shifts have correct shape: ({embedding_dim},)")

# Verify control means have correct dimensions
print(f"\nVerifying control mean dimensions...")
for cell_type, mean in shift_table.control_means.items():
    assert mean.shape == (embedding_dim,), f"Control mean for {cell_type} has wrong shape: {mean.shape}"
print(f"✓ All {n_control_means} control means have correct shape: ({embedding_dim},)")

# Show example shifts
print(f"\nExample shifts:")
for i, (key, shift) in enumerate(list(shift_table.shifts.items())[:3]):
    cell_type, pert = key
    magnitude = np.linalg.norm(shift)
    n_control, n_pert = shift_table.n_samples[key]
    print(f"\n  {i+1}. Cell type: {cell_type}")
    print(f"     Perturbation: {pert}")
    print(f"     Shift magnitude: {magnitude:.4f}")
    print(f"     Sample sizes: {n_control} control, {n_pert} perturbed")
    print(f"     First 5 shift values: {shift[:5]}")

print("\n✓ All assertions passed!")

## Step 4: Analyze Shift Statistics

Let's examine the distribution of shift magnitudes across different perturbations.

In [ ]:
# Get statistics
stats_df = shift_table.get_statistics()

print("Shift magnitude statistics:")
print(stats_df['shift_magnitude'].describe())

print("\nTop 20 perturbations by shift magnitude:")
print(stats_df.head(20))

# Save statistics
stats_path = OUTPUT_DIR / "shift_statistics.csv"
stats_df.to_csv(stats_path, index=False)
print(f"\nStatistics saved to: {stats_path}")

In [ ]:
# Plot distribution of shift magnitudes
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Histogram
axes[0].hist(stats_df['shift_magnitude'], bins=50, edgecolor='black', alpha=0.7)
axes[0].axvline(stats_df['shift_magnitude'].median(), color='r', linestyle='--', 
                label=f'Median: {stats_df["shift_magnitude"].median():.4f}')
axes[0].set_xlabel('Shift Magnitude (L2 Norm)', fontsize=12)
axes[0].set_ylabel('Count', fontsize=12)
axes[0].set_title('Distribution of Shift Magnitudes', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Boxplot by cell type
stats_df.boxplot(column='shift_magnitude', by='cell_type', ax=axes[1])
axes[1].set_xlabel('Cell Type', fontsize=12)
axes[1].set_ylabel('Shift Magnitude', fontsize=12)
axes[1].set_title('Shift Magnitudes by Cell Type', fontsize=14)
plt.suptitle('')  # Remove default title

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "shift_magnitude_distribution.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"Plot saved to: {OUTPUT_DIR / 'shift_magnitude_distribution.png'}")

## Step 5: Load Test Data

In [ ]:
print("Loading test data...")
adata_test = sc.read_h5ad(TEST_DATA_PATH)

print(f"\nTest data shape: {adata_test.shape}")
print(f"  - Cells: {adata_test.n_obs:,}")
print(f"  - Features: {adata_test.n_vars:,}")

# Verify test data structure matches training data
assert CELL_TYPE_COL in adata_test.obs.columns, f"Column '{CELL_TYPE_COL}' not found in test data"
assert PERT_COL in adata_test.obs.columns, f"Column '{PERT_COL}' not found in test data"

# Get test embeddings
if EMBED_KEY and EMBED_KEY in adata_test.obsm:
    test_embeddings = adata_test.obsm[EMBED_KEY]
else:
    test_embeddings = adata_test.X.toarray() if hasattr(adata_test.X, 'toarray') else adata_test.X

print(f"Test embedding shape: {test_embeddings.shape}")
assert test_embeddings.shape[1] == embedding_dim, f"Test embedding dim {test_embeddings.shape[1]} doesn't match train {embedding_dim}"

# Test set statistics
test_cell_types = adata_test.obs[CELL_TYPE_COL].unique()
test_perts = adata_test.obs[PERT_COL].unique()
n_test_control = (adata_test.obs[PERT_COL] == CONTROL_PERT).sum()
n_test_perturbed = (adata_test.obs[PERT_COL] != CONTROL_PERT).sum()

print(f"\nTest set statistics:")
print(f"  Cell types: {len(test_cell_types)}")
print(f"  Perturbations: {len(test_perts)}")
print(f"  Control cells: {n_test_control:,} ({n_test_control/adata_test.n_obs*100:.1f}%)")
print(f"  Perturbed cells: {n_test_perturbed:,} ({n_test_perturbed/adata_test.n_obs*100:.1f}%)")

print("\n✓ Test data loaded and verified!")

## Step 6: Evaluate Mean Shift Baseline

Now we apply the mean shift to test the prediction accuracy.

**For each (cell_type, perturbation) combination:**
1. Get all control cells of that cell type
2. Apply the precomputed mean shift: `predicted = control_embedding + shift`
3. Compare predictions to actual perturbed cells
4. Compute MSE loss

This tests whether the shift generalizes from training data to new cells.

In [ ]:
results = evaluate_mean_shift_baseline(
    adata_test_path=TEST_DATA_PATH,
    shift_table=shift_table,
    control_pert=CONTROL_PERT,
    cell_type_col=CELL_TYPE_COL,
    pert_col=PERT_COL,
    embed_key=EMBED_KEY
)

# Save results
results_path = OUTPUT_DIR / "mean_shift_results.pkl"
with open(results_path, 'wb') as f:
    pickle.dump(results, f)

print(f"\nResults saved to: {results_path}")

## Step 7: Verify Prediction Shapes

In [ ]:
print("="*60)
print("PREDICTION SHAPE VERIFICATION")
print("="*60)

predictions = results['predictions']
ground_truth = results['ground_truth']

print(f"\nPredictions shape: {predictions.shape}")
print(f"Ground truth shape: {ground_truth.shape}")

# Verify shapes match
assert predictions.shape == ground_truth.shape, "Prediction and ground truth shapes don't match!"
assert predictions.shape[1] == embedding_dim, f"Prediction embedding dim {predictions.shape[1]} doesn't match expected {embedding_dim}"

print(f"\n✓ Shapes verified!")
print(f"  - Number of predictions: {predictions.shape[0]:,}")
print(f"  - Embedding dimension: {predictions.shape[1]:,}")

# Check for NaN or Inf values
assert not np.any(np.isnan(predictions)), "Predictions contain NaN values!"
assert not np.any(np.isinf(predictions)), "Predictions contain Inf values!"
assert not np.any(np.isnan(ground_truth)), "Ground truth contains NaN values!"
assert not np.any(np.isinf(ground_truth)), "Ground truth contains Inf values!"

print(f"\n✓ No NaN or Inf values detected!")

## Step 8: Analyze Results

### Overall Performance

In [ ]:
overall_mse = results['overall_mse']
n_predictions = results['n_predictions']

print("="*60)
print("MEAN SHIFT BASELINE PERFORMANCE")
print("="*60)

print(f"\nOverall MSE: {overall_mse:.6f}")
print(f"Number of predictions: {n_predictions:,}")
print(f"RMSE: {np.sqrt(overall_mse):.6f}")

# Compute additional metrics
mae = np.mean(np.abs(predictions - ground_truth))
print(f"Mean Absolute Error (MAE): {mae:.6f}")

# Cosine similarity
from scipy.spatial.distance import cosine
cosine_similarities = []
for i in range(len(predictions)):
    sim = 1 - cosine(predictions[i], ground_truth[i])
    cosine_similarities.append(sim)

mean_cosine_sim = np.mean(cosine_similarities)
print(f"Mean Cosine Similarity: {mean_cosine_sim:.4f}")

### Per Cell Type Performance

In [ ]:
per_ct_mse = results['per_celltype_mse']

print("\nPer Cell Type MSE:")
for ct, mse in sorted(per_ct_mse.items(), key=lambda x: x[1]):
    n = np.sum(np.array(results['cell_types']) == ct)
    print(f"  {ct}: {mse:.6f} (n={n:,})")

# Plot
fig, ax = plt.subplots(figsize=(10, 6))
cell_types = list(per_ct_mse.keys())
mse_values = [per_ct_mse[ct] for ct in cell_types]

ax.barh(cell_types, mse_values, color='steelblue', alpha=0.7)
ax.set_xlabel('MSE', fontsize=12)
ax.set_ylabel('Cell Type', fontsize=12)
ax.set_title('Mean Shift Performance by Cell Type', fontsize=14)
ax.grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "per_celltype_performance.png", dpi=150, bbox_inches='tight')
plt.show()

### Per Perturbation Performance

In [ ]:
per_pert_mse = results['per_perturbation_mse']

print("\nTop 10 perturbations by MSE (worst performance):")
sorted_perts = sorted(per_pert_mse.items(), key=lambda x: x[1], reverse=True)
for pert, mse in sorted_perts[:10]:
    n = np.sum(np.array(results['perturbations']) == pert)
    print(f"  {pert}: {mse:.6f} (n={n:,})")

print("\nBottom 10 perturbations by MSE (best performance):")
for pert, mse in sorted_perts[-10:]:
    n = np.sum(np.array(results['perturbations']) == pert)
    print(f"  {pert}: {mse:.6f} (n={n:,})")

## Step 9: Visualize Prediction Quality

In [ ]:
# Scatter plot: predicted vs actual (first dimension)
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Plot for first dimension
axes[0].scatter(ground_truth[:, 0], predictions[:, 0], alpha=0.3, s=1)
axes[0].plot([ground_truth[:, 0].min(), ground_truth[:, 0].max()],
             [ground_truth[:, 0].min(), ground_truth[:, 0].max()],
             'r--', label='Perfect prediction')
axes[0].set_xlabel('Ground Truth (Dimension 0)', fontsize=12)
axes[0].set_ylabel('Predicted (Dimension 0)', fontsize=12)
axes[0].set_title('Prediction vs Ground Truth (Dim 0)', fontsize=14)
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Residual plot
residuals = predictions - ground_truth
axes[1].hist(residuals.flatten(), bins=100, edgecolor='black', alpha=0.7)
axes[1].axvline(0, color='r', linestyle='--', label='Zero error')
axes[1].set_xlabel('Residual (Predicted - Ground Truth)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title('Distribution of Residuals', fontsize=14)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "prediction_quality.png", dpi=150, bbox_inches='tight')
plt.show()

## Summary and Next Steps

### Key Findings

1. **Mean Shift Baseline MSE:** [will be filled in after running]
2. **Number of (cell_type, perturbation) pairs evaluated:** [will be filled in]
3. **Best performing cell types:** [will be filled in]
4. **Worst performing perturbations:** [will be filled in]

### Next Steps

1. **Compare to State Transformer:**
   - Run State model training with checkpoints at multiple epochs
   - Evaluate State model on same test set
   - Compare MSE values to determine when transformer surpasses mean shift

2. **Analyze Failure Cases:**
   - Identify perturbations where mean shift performs poorly
   - Investigate whether these perturbations have high cell-to-cell variability

3. **Test Initialization:**
   - Initialize State transformer with mean shift weights
   - Compare training convergence speed



## Save Final Summary

In [ ]:
summary = {
    'overall_mse': overall_mse,
    'n_predictions': n_predictions,
    'rmse': np.sqrt(overall_mse),
    'mae': mae,
    'mean_cosine_similarity': mean_cosine_sim,
    'n_shifts_computed': len(shift_table.shifts),
    'n_cell_types': len(unique_cell_types),
    'n_perturbations': len(unique_perts),
    'embedding_dim': embedding_dim
}

summary_df = pd.DataFrame([summary])
summary_path = OUTPUT_DIR / "summary.csv"
summary_df.to_csv(summary_path, index=False)

print("\n" + "="*60)
print("ANALYSIS COMPLETE!")
print("="*60)
print(f"\nAll results saved to: {OUTPUT_DIR}")
print(f"\nFiles created:")
for f in OUTPUT_DIR.glob("*"):
    print(f"  - {f.name}")